# EDA — Telco Customer Churn

Esta análise exploratória documenta a qualidade dos dados, a distribuição do target e os principais padrões associados ao churn. Nenhuma decisão de modelagem deve ser tomada antes desta etapa.

**Fonte:** IBM Telco Customer Churn / Kaggle. O dataset possui 7.043 clientes e 21 colunas.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data.download import download_dataset

DATA_PATH = download_dataset()
df = pd.read_csv(DATA_PATH)
df.head()

## 1. Dimensão e estrutura

In [ ]:
print('Shape:', df.shape)
print('Duplicated rows:', df.duplicated().sum())
print('Unique customer IDs:', df['customerID'].nunique())
df.info()

In [ ]:
summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'n_unique': df.nunique(),
}).sort_values(['missing_pct', 'n_unique'], ascending=[False, True])
summary

## 2. Tratamento inicial a investigar

`TotalCharges` chega como texto no arquivo original. Antes de convertê-la para numérico, contabilizamos strings vazias/espaços para não mascarar um problema de qualidade de dados.

In [ ]:
total_charges_raw = df['TotalCharges'].astype(str)
blank_total_charges = total_charges_raw.str.strip().eq('').sum()
print('TotalCharges vazios:', blank_total_charges)

df['TotalCharges_numeric'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('NaN após conversão:', df['TotalCharges_numeric'].isna().sum())

df.loc[df['TotalCharges_numeric'].isna(), ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

## 3. Target: Churn

Vamos verificar a proporção de classes. O desbalanceamento será considerado na validação e na escolha das métricas.

In [ ]:
target_dist = (df['Churn'].value_counts(dropna=False).rename_axis('Churn').to_frame('count'))
target_dist['pct'] = target_dist['count'] / len(df) * 100
target_dist

In [ ]:
sns.countplot(data=df, x='Churn')
plt.title('Distribuição do target — Churn')
plt.tight_layout()
plt.show()

## 4. Variáveis numéricas

`tenure`, `MonthlyCharges` e `TotalCharges` são os principais campos quantitativos. A relação entre eles e churn será explorada sem eliminar outliers automaticamente.

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges_numeric']
df[numeric_cols].describe().T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(data=df, x='Churn', y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 5. Variáveis categóricas e churn rate

Para cada variável categórica, calculamos a taxa de churn por categoria. Isso orientará o feature engineering e a interpretação do modelo.

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols.remove('customerID')

for col in categorical_cols:
    rates = (df.groupby(col, dropna=False)['Churn']
             .apply(lambda s: (s == 'Yes').mean() * 100)
             .sort_values(ascending=False))
    print(f'\n### {col}')
    display(rates.round(2).to_frame('churn_pct'))

## 6. Relações de negócio prioritárias

Contrato, serviço de internet, método de pagamento e tenure são candidatos naturais a investigação porque representam vínculo, oferta e comportamento de permanência.

In [ ]:
business_cols = ['Contract', 'InternetService', 'PaymentMethod', 'PaperlessBilling']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.ravel(), business_cols):
    rates = df.groupby(col)['Churn'].apply(lambda s: (s == 'Yes').mean() * 100).sort_values(ascending=False)
    sns.barplot(x=rates.values, y=rates.index, ax=ax)
    ax.set_title(f'Churn por {col}')
    ax.set_xlabel('Churn (%)')
    ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
df['tenure_bin'] = pd.cut(
    df['tenure'],
    bins=[-1, 3, 6, 12, 24, 48, 72],
    labels=['0-3', '4-6', '7-12', '13-24', '25-48', '49-72'],
)
tenure_churn = df.groupby('tenure_bin', observed=False)['Churn'].apply(lambda s: (s == 'Yes').mean() * 100)
tenure_churn.round(2).to_frame('churn_pct')

## 7. Hipóteses para modelagem

Ao final da EDA, registrar hipóteses verificáveis em vez de conclusões causais. Exemplos: clientes em contratos mensais podem apresentar maior churn; menor tenure pode estar associado a maior risco; determinados métodos de pagamento podem concentrar churn.

Essas hipóteses serão testadas pelos modelos e pelas métricas, não tratadas como causalidade.

## 8. Próximos passos

1. Definir o tratamento de `TotalCharges` com base na evidência encontrada.
2. Separar treino/teste de forma estratificada antes de aprender transformações.
3. Construir um `ColumnTransformer` dentro de um `Pipeline`.
4. Treinar Regressão Logística como baseline.
5. Comparar Random Forest e `MLPClassifier` com validação cruzada.
